In [94]:
import pandas as pd
import numpy as np
import logging

In [95]:
# Creating an e-commerce transaction messy dataset
ecommerce_data = {
    'order_id': [ord for ord in range(501, 511)],
    'customer_email': ['  ANNA@gmail.com', 'bob.builder@yahoo.com', 'charlie@outlook,com', 'DIANA@GMAIL.COM', '  ', 'eddie@aol.com', 'fiona@icloud.com', 'bob.builder@yahoo.com', 'george@gmail.com', 'hannah@work.net'], # Typo in comma, whitespace, empty string, duplicates
    'item_purchased': ['Wireless Mouse', 'Mechanical Keyboard', 'USB-C Hub ', 'Monitor 27"', 'Mechanical Keyboard', 'Webcam HD', 'Wireless Mouse', 'Mechanical Keyboard', 'Desk Mat', 'Desk Mat'],
    'quantity': ['1', '2', 'one', '3', '2', np.nan, '1', '2', '4', 'four'], # String numbers and words ('one', 'four') mixed with integers and NaN
    'unit_price': ['$25.50', ' $120.00 ', '45.00USD', '$300', '$ 120.00 ', '50.00', '25.50', ' $120.00 ', '15.00', '15.00'], # Currency codes, mixed symbols, spaces
    'shipping_status': ['Delivered', 'Pending', 'Delivered', 'Cancelled', 'Pending', 'Delivered', 'delivered', 'Pending', 'Shipped ', 'Shipped'] # Inconsistent casing and trailing spaces
}

df_shop = pd.DataFrame(ecommerce_data)
df_shop.to_csv('shop.csv',index = False)

In [108]:
import logging
import os
import pandas as pd
import numpy as np

# Ensure root logger handlers are cleared to allow basicConfig to re-run effectively
# This is a common workaround for re-executing cells in interactive environments.
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    handler.close()

# Now basicConfig will effectively set up new handlers
logging.basicConfig(
    level = logging.INFO,
    format = '%(asctime)s:%(levelname)s:%(message)s',
    handlers= [logging.FileHandler('shops1.txt', mode='w'), # Use 'w' mode to overwrite existing logs
              logging.StreamHandler()
              ])

# Get the logger associated with this context
logger = logging.getLogger(__name__)
logger.info(f'Start data pipeline process')

def data(file_name):
    try:
      logger.info(f'Start Reading the file {file_name}')
      df = pd.read_csv(file_name)
      logger.info(f'Replacing white space and correcting email format')
      df.loc[:,'customer_email'] = df['customer_email'].replace(r'^\s*$',np.nan,regex = True)
      df = df.dropna(subset = ['customer_email'])
      df.loc[:,'customer_email'] = df['customer_email'].str.lower()
      # Correcting the email domain typo
      df.loc[:,'customer_email'] = df['customer_email'].str.replace('@outlook,com', '@outlook.com', regex=False)
      logger.info(f'perform trimming')
      df.loc[:,'item_purchased'] = df['item_purchased'].str.strip()
      logger.info(f'perform data type conversion object into integer')
      df.loc[:,'quantity'] = df['quantity'].replace('one','1')
      df.loc[:,'quantity'] = df['quantity'].replace('four','4')
      df.loc[:,'quantity'] = df['quantity'].fillna('0')
      df.loc[:,'quantity'] = df['quantity'].astype('int')
      logger.info(f'perform data type conversion object into float')
      df.loc[:,'unit_price'] = df['unit_price'].str.replace('$',' ',regex=False)
      df.loc[:,'unit_price'] = df['unit_price'].str.replace('USD',' ',regex=False)
      df.loc[:,'unit_price'] = df['unit_price'].str.strip()
      df.loc[:,'unit_price'] = df['unit_price'].astype('float')
      logger.info(f'Setting proper categorical values for shipping status')
      # Standardize shipping_status: remove spaces, lowercase, then capitalize first letter
      df['shipping_status'] = df['shipping_status'].str.strip().str.lower().str.capitalize()
      logger.info(f'Data cleaning completed successfully.')
      return df
    except Exception as e:
        logger.exception(f'Error Occurred during data cleaning: {e}')
        # Force a flush right now so the error gets written to the file
        for handler in logging.root.handlers:
            handler.flush()
        raise # Re-raise the exception after logging it

2026-08-15 11:39:39,366:INFO:Start data pipeline process


In [110]:
# Re-execute the data function to regenerate logs with the corrected logging setup
df = data('shop.csv')
# Explicitly flush the root logger's file handler to ensure logs are written before reading
for handler in logging.root.handlers:
    if isinstance(handler, logging.FileHandler):
        handler.flush()
display(df.head())

2026-08-15 11:39:40,580:INFO:Start Reading the file shop.csv
2026-08-15 11:39:40,585:INFO:Replacing white space and correcting email format
2026-08-15 11:39:40,593:INFO:perform trimming
2026-08-15 11:39:40,596:INFO:perform data type conversion object into integer
2026-08-15 11:39:40,601:INFO:perform data type conversion object into float
2026-08-15 11:39:40,607:INFO:Setting proper categorical values for shipping status
2026-08-15 11:39:40,611:INFO:Data cleaning completed successfully.


,order_id,customer_email,item_purchased,quantity,unit_price,shipping_status
0,501,anna@gmail.com,Wireless Mouse,1,25.5,Delivered
1,502,bob.builder@yahoo.com,Mechanical Keyboard,2,120.0,Pending
2,503,charlie@outlook.com,USB-C Hub,1,45.0,Delivered
3,504,diana@gmail.com,"Monitor 27""",3,300.0,Cancelled
5,506,eddie@aol.com,Webcam HD,0,50.0,Delivered


In [111]:
# Now read and print the log file to verify the content
with open('/content/shops1.txt', 'r') as f:
    logs = f.read()
print(logs)

2026-08-15 11:39:39,366:INFO:Start data pipeline process
2026-08-15 11:39:39,961:INFO:Start Reading the file shop.csv
2026-08-15 11:39:39,966:INFO:Replacing white space and correcting email format
2026-08-15 11:39:39,972:INFO:perform trimming
2026-08-15 11:39:39,974:INFO:perform data type conversion object into integer
2026-08-15 11:39:39,978:INFO:perform data type conversion object into float
2026-08-15 11:39:39,981:INFO:Setting proper categorical values for shipping status
2026-08-15 11:39:39,984:INFO:Data cleaning completed successfully.
2026-08-15 11:39:40,580:INFO:Start Reading the file shop.csv
2026-08-15 11:39:40,585:INFO:Replacing white space and correcting email format
2026-08-15 11:39:40,593:INFO:perform trimming
2026-08-15 11:39:40,596:INFO:perform data type conversion object into integer
2026-08-15 11:39:40,601:INFO:perform data type conversion object into float
2026-08-15 11:39:40,607:INFO:Setting proper categorical values for shipping status
2026-08-15 11:39:40,611:INFO:D

In [97]:
df=data('shop.csv')

In [109]:
df = data('shop.csv')
# Explicitly flush the root logger's file handler to ensure logs are written before reading
for handler in logging.root.handlers:
    if isinstance(handler, logging.FileHandler):
        handler.flush()
display(df.head())

2026-08-15 11:39:39,961:INFO:Start Reading the file shop.csv
2026-08-15 11:39:39,966:INFO:Replacing white space and correcting email format
2026-08-15 11:39:39,972:INFO:perform trimming
2026-08-15 11:39:39,974:INFO:perform data type conversion object into integer
2026-08-15 11:39:39,978:INFO:perform data type conversion object into float
2026-08-15 11:39:39,981:INFO:Setting proper categorical values for shipping status
2026-08-15 11:39:39,984:INFO:Data cleaning completed successfully.


,order_id,customer_email,item_purchased,quantity,unit_price,shipping_status
0,501,anna@gmail.com,Wireless Mouse,1,25.5,Delivered
1,502,bob.builder@yahoo.com,Mechanical Keyboard,2,120.0,Pending
2,503,charlie@outlook.com,USB-C Hub,1,45.0,Delivered
3,504,diana@gmail.com,"Monitor 27""",3,300.0,Cancelled
5,506,eddie@aol.com,Webcam HD,0,50.0,Delivered


In [107]:
with open('/content/shops1.txt', 'r') as f:
    logs = f.read()
print(logs)

# Removed logging.shutdown() here to avoid prematurely closing loggers.
# The system typically handles logging shutdown when the environment ends.